# ForgeEdge — Rule Registry (Modulo 4)

Questo notebook mostra l'utilizzo del modulo **RuleRegistry**, il quarto e ultimo passo della pipeline FORGE.

```
Market Context  →  Event Discovery  →  Alpha Discovery  →  Rule Discovery  →  Rule Registry
```

Rule Registry riceve le **regole validate** da Rule Discovery — un pool per ogni ticker della sessione — e le raccoglie in un **registro in-memory** (stateless: ricostruito da zero a ogni sessione). Risponde alla domanda: *tra tutte le regole validate, quali sono distinte, complementari e generiche su più ticker?*

I cinque step del modulo:

1. **Ingestion** — un documento per ogni regola `EDGE` / `PARTIAL-EDGE`.
2. **Matrici di correlazione** — Jaccard sulle attivazioni (per data) e Spearman sui gain.
3. **Deduplicazione** — marca (non elimina) i duplicati; mantiene le catene.
4. **Cross-ticker backtest** — ogni regola testata su tutti gli altri ticker, con le soglie ricalcolate sulla distribuzione locale → badge `GENERIC` / `PARTIAL` / `SPECIFIC` / `ISOLATED`.
5. **Export** — tabella piatta (CSV / Excel) + report HTML autocontenuto.

> Per rendere il notebook **eseguibile ovunque**, costruiamo prima una sessione **sintetica** a 3 ticker con regole già validate (così ogni funzionalità — dedup, cross-ticker, badge — è visibile). In coda mostriamo anche il wiring reale `RuleRegistry.from_forge_results(...)` partendo da un Excel multi-ticker, se presente.

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, "../src")   # per esecuzione da notebooks/

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from forgedge import forge
from forgedge.event_discovery import DiscoveryConfig
from forgedge.event_discovery.models import (
    ActivationStats, EventCandidate, EventComponent, GateResult,
)
from forgedge.rule_discovery import BacktestParams, run_backtest
from forgedge.rule_discovery.models import RuleDiscoveryResponse, ValidatedRule
from forgedge.rule_registry import RuleRegistry, RuleSubmission, RegistryConfig

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 80)

## 1. Una sessione FORGE sintetica a 3 ticker

Generiamo tre KPI Table (ADAUSDC, SOLUSDC, BTCUSDC) con due feature e due pattern dal comportamento opposto:

* **`feat`** — pattern **universale**: un `feat` basso innesca un rialzo localizzato (i `k` barre successivi) *su tutti i ticker*. Le regole su `feat` generalizzano → `GENERIC`.
* **`feat2`** — pattern **specifico**: un `feat2` alto innesca un rialzo **solo su BTCUSDC** (su ADA/SOL è rumore). Una regola su `feat2` passa sul ticker sorgente ma fallisce altrove → `SPECIFIC` / `ISOLATED`.

Fuori dai pattern il prezzo ha un lento *bleed* negativo: un ingresso casuale, non sincronizzato con un pattern, perde. I range di `feat` differiscono per ticker, così la **ricalibrazione delle soglie** cross-ticker fa un lavoro reale (la stessa percentile mappa su soglie assolute diverse).

In [ ]:
def make_frame(seed, feat_lo, feat_hi, feat2=False, n=3000, g=0.012, b=0.0015, k=5, tq=0.03):
    """OHLC sintetico.

    feat basso (< percentile tq)  → rialzo localizzato +g per k barre (universale).
    feat2 alto (> 1-tq)           → stesso rialzo, ma solo se feat2=True (specifico del ticker).
    Fuori dai pattern: bleed -b/barra → gli ingressi non allineati a un pattern perdono.
    Ordini a mercato, così l'ingresso del pattern cattura sempre il movimento (nessun fill mancato).
    """
    rng = np.random.default_rng(seed)
    feat  = rng.uniform(feat_lo, feat_hi, n)
    feat2_s = rng.uniform(0.0, 1.0, n)
    qf = np.quantile(feat, tq)
    q2 = np.quantile(feat2_s, 1.0 - tq)
    ret = np.full(n, -b) + rng.normal(0.0, 0.001, n)
    for i in range(n):
        if feat[i] < qf:                         # pattern universale
            ret[i + 1: min(i + 1 + k, n)] = g
        if feat2 and feat2_s[i] > q2:            # pattern specifico del ticker
            ret[i + 1: min(i + 1 + k, n)] = g
    close = 100.0 * np.cumprod(1.0 + ret)
    return pd.DataFrame({
        "open_dt": pd.date_range("2024-01-01", periods=n, freq="1h"),
        "open":  close * (1.0 + rng.normal(0.0, 0.0005, n)),
        "high":  close * 1.003,
        "low":   close * 0.997,
        "close": close,
        "volume": np.abs(rng.normal(1e6, 1e5, n)),
        "feat":  feat,
        "feat2": feat2_s,
    })

frames = {
    "ADAUSDC": make_frame(seed=1, feat_lo=0.0, feat_hi=1.0),
    "SOLUSDC": make_frame(seed=2, feat_lo=0.0, feat_hi=2.0),               # feat su scala diversa
    "BTCUSDC": make_frame(seed=3, feat_lo=0.0, feat_hi=0.5, feat2=True),   # feat2 predittivo solo qui
}

{t: f.shape for t, f in frames.items()}

## 2. Costruire l'input del registro: `RuleSubmission`

Rule Registry consuma una lista di `RuleSubmission` — ognuna lega una `RuleDiscoveryResponse` validata al suo **ticker sorgente** e al suo **Event Candidate** (necessario per ricostruire il segnale, con soglie ricalibrate, sugli *altri* ticker).

Qui le costruiamo a mano (in produzione arrivano da Rule Discovery). Definiamo due helper: uno per fabbricare un `EventCandidate` a una componente (`col < soglia` o `col > soglia`) e uno per produrre la `RuleDiscoveryResponse` con un backtest reale sul ticker sorgente.

In [ ]:
def make_candidate(event_id, col, thr, direction="below"):
    op = "<" if direction == "below" else ">"
    comp = EventComponent(
        source_feature=col, transform="identity", transform_params={},
        transformed_col=col, threshold=thr, threshold_type="distributional",
        direction=direction, event_type="threshold",
        expression=f"{col} {op} {thr:.4g}", source_cols=[],
    )
    stats = ActivationStats(100, 12, 0, 0.2, 8.0)
    gate  = GateResult(True, 100, 12, 0.2, 8.0)
    return EventCandidate(event_id, "CANDIDATE", [comp], comp.expression, stats, gate)

PARAMS = BacktestParams(buy_type="market", sell_pct=0.03, target_h=6, fee=0.001)

def make_response(ticker, cand, verdict="EDGE", alpha_id="ALPHA"):
    f = frames[ticker].copy()
    f.index = pd.DatetimeIndex(pd.to_datetime(f["open_dt"]))
    f["__s__"] = cand.apply(f).fillna(0).to_numpy()
    summary = run_backtest(f, "__s__", PARAMS, timestamp_col="open_dt")
    return RuleDiscoveryResponse(
        date="2026-06-14", verdict=verdict, alpha_id=alpha_id,
        asset=ticker, timeframe="1H",
        validated_rule=ValidatedRule(cand.expression, cand.event_id, PARAMS),
        in_sample_summary=summary, walk_forward=None,
        statistical_validation=None, regime_analysis=None,
    )

def make_submission(ticker, col, q, direction="below", event_id=None, alpha_id=None):
    thr = float(np.quantile(frames[ticker][col], q))
    event_id = event_id or f"EVT-{ticker}-{col}-{int(q*100)}"
    alpha_id = alpha_id or f"ALPHA-{ticker}-{col}"
    cand = make_candidate(event_id, col, thr, direction)
    return RuleSubmission(ticker, make_response(ticker, cand, alpha_id=alpha_id), cand)

submissions = [
    # ADA: due regole quasi identiche sullo stesso pattern → una verrà marcata duplicata
    make_submission("ADAUSDC", "feat", 0.03, alpha_id="ALPHA-ADA-1"),
    make_submission("ADAUSDC", "feat", 0.04, alpha_id="ALPHA-ADA-2"),
    # SOL: stesso pattern universale, ticker diverso
    make_submission("SOLUSDC", "feat", 0.03, alpha_id="ALPHA-SOL-1"),
    # BTC: pattern specifico del ticker (feat2 alto) → non generalizza
    make_submission("BTCUSDC", "feat2", 0.97, direction="above", alpha_id="ALPHA-BTC-1"),
]

for s in submissions:
    print(f"{s.ticker:<9} {s.response.verdict:<6} {s.response.validated_rule.expression}")

## 3. Eseguire il registro

`RuleRegistry(submissions, frames, config).run()` esegue in sequenza i 4 step (ingestion → correlazioni → deduplicazione → cross-ticker). L'export è on-demand.

I default delle soglie (lasciate `TBD` nella specifica) sono calibrati sull'esempio della documentazione: `overlap_threshold=0.70`, `cross_pf_threshold=2.0`, `generic_ratio_threshold=2/3`.

In [ ]:
config = RegistryConfig(
    overlap_threshold=0.70,
    cross_pf_threshold=2.0,
    generic_ratio_threshold=2/3,
    export_format="csv",   # CSV per non richiedere openpyxl in questo notebook
)

registry = RuleRegistry(submissions, frames, config).run()
registry.summary()

## 4. Step 1 — Ingestion

Ogni regola validata diventa un **documento** con id `RULE_<TICKER>_<NN>`. Gli array paralleli `activation_idx` / `activation_dates` / `gains` (recuperati replicando il backtest sul ticker sorgente) sono la struttura dati chiave per le matrici di correlazione.

In [ ]:
d = registry.documents[0]
print("rule_id        :", d.rule_id)
print("source_ticker  :", d.source_ticker)
print("expression     :", d.expression)
print("grade / verdict:", d.grade, "/", d.verdict)
print("n. trade       :", len(d.gains))
print("prime 3 date   :", d.activation_dates[:3])
print("primi 3 gain   :", [round(g, 4) for g in d.gains[:3]])
print("stats          :", {k: d.stats[k] for k in ('pf', 'win_rate', 'total_trades', 'dsr')})

## 5. Step 2 — Matrici di correlazione

**Jaccard** misura la sovrapposizione temporale delle attivazioni (confronto *per data*, così regole di ticker diversi sono comparabili). **Spearman** misura la correlazione dei gain allineati per data (le barre senza trade valgono `0`).

> Spearman è calcolata senza `scipy` (rank + Pearson) per mantenere il runtime `numpy`/`pandas`-only.

In [ ]:
m = registry.matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
specs = [(m.jaccard, "Jaccard — overlap attivazioni", "Blues", 0.0, 1.0),
         (m.spearman, "Spearman — correlazione gain", "RdYlGn", -1.0, 1.0)]
for ax, (mat, title, cmap, vmin, vmax) in zip(axes, specs):
    im = ax.imshow(mat.values, vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_xticks(range(len(m.rule_ids))); ax.set_xticklabels(m.rule_ids, rotation=45, ha="right")
    ax.set_yticks(range(len(m.rule_ids))); ax.set_yticklabels(m.rule_ids)
    ax.set_title(title)
    for i in range(len(m.rule_ids)):
        for j in range(len(m.rule_ids)):
            ax.text(j, i, f"{mat.values[i, j]:.2f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

## 6. Step 3 — Deduplicazione

Per ogni coppia con Jaccard sopra soglia, la regola con PF minore è marcata `is_duplicate=True` e `duplicate_of` punta alla **dominante immediata** (le catene `R03 → R02 → R01` sono preservate). Nessuna regola viene eliminata — l'utente vede tutto.

Le due regole ADA sullo stesso pattern (`feat`) hanno attivazioni quasi identiche → una è marcata duplicata.

In [ ]:
registry.summary()[["rule_id", "source_ticker", "pf", "overlap_max",
                     "gain_corr_max", "is_duplicate", "duplicate_of"]]

## 7. Step 4 — Cross-ticker backtest

Ogni regola è testata su tutti gli altri ticker. La struttura logica resta invariata; le **soglie assolute** vengono ricalcolate sul percentile corrispondente della distribuzione del ticker target (`expression_adapted`). Il PF sul target determina `PASS`/`FAIL`, e la frazione di PASS dà il badge di genericità.

In [ ]:
for d in registry.documents:
    print(f"{d.rule_id}  [{d.classification}]  score {d.cross_ticker_score}/{d.cross_ticker_total}")
    for t, r in d.cross_ticker.items():
        flag = "✅" if r.verdict == "PASS" else "❌"
        print(f"    {flag} {t:<9} PF {r.pf:>7.2f}  WR {r.win_rate:6.2%}  T {r.total_trades:>3}"
              f"   adapted: {r.expression_adapted}")
    print()

Le regole sul pattern **universale** (`feat`) generalizzano su tutti i ticker → **GENERIC**; la regola sul pattern **specifico** di BTC (`feat2`) fallisce altrove → **SPECIFIC** / **ISOLATED**. Esattamente la proprietà che il Rule Registry *misura a posteriori*, senza mescolare i dati a monte.

## 8. Step 5 — Export

Due artefatti per sessione: la **tabella piatta** (una riga per regola, blocco di colonne `pf_{TICKER}` / `wr_{TICKER}` / `verdict_{TICKER}` per ogni ticker) e il **report HTML** autocontenuto.

In [ ]:
flat = registry.flat_table()
print("shape:", flat.shape)
flat

In [ ]:
path = registry.export("forge_flat_table.csv")
print("Tabella piatta scritta in:", path)

### Report HTML autocontenuto

SVG inline (heatmap delle correlazioni, equity curve per regola), badge di genericità, banner sui duplicati, cross-ticker summary e trade log. Nessuna CDN, nessun JavaScript esterno — si apre offline in qualsiasi browser. Qui lo incorporiamo direttamente nel notebook.

In [ ]:
html = registry.html_report(timeframe="1H")
with open("forge_report.html", "w", encoding="utf-8") as fh:
    fh.write(html)
print(f"Report scritto in forge_report.html ({len(html):,} bytes)")
display(HTML(html))

## 9. Wiring reale — `RuleRegistry.from_forge_results`

In una sessione reale si esegue la pipeline `forge(...)` **separatamente per ogni ticker** (la generalizzabilità è misurata dopo, non costruita mescolando i dati) e si passa il dizionario `{ticker: ForgeResult}` a `from_forge_results`, che estrae regole validate, Event Candidate e grade automaticamente.

La cella sotto gira solo se è presente un Excel multi-ticker in `../data/test1h.xlsx`.

In [ ]:
import os
DATA_PATH = "../data/test1h.xlsx"

if os.path.exists(DATA_PATH):
    raw = pd.read_excel(DATA_PATH)
    raw["open_dt"] = pd.to_datetime(raw["open_time"], unit="ms")
    results = {}
    for ticker in sorted(raw["symbol"].unique()):
        d = raw[raw["symbol"] == ticker].copy().sort_values("open_dt")
        d = d.drop(columns=[c for c in ("symbol", "timeframe", "open_time") if c in d.columns])
        d = d.dropna().reset_index(drop=True)
        results[ticker] = forge(d, asset=ticker, timeframe="1H",
                                event_discovery_config=DiscoveryConfig(timestamp_col="open_dt"))
        print(f"  {ticker:<10}: {len(results[ticker].edges())} regole validate")

    real_registry = RuleRegistry.from_forge_results(
        results, RegistryConfig(export_format="csv")
    ).run()
    display(real_registry.summary())
else:
    print("Dataset reale non trovato in", DATA_PATH,
          "— uso la demo sintetica delle sezioni precedenti.")